# SR (direct) vs Z (ADMM) — Bpp / PSNR

Fair comparison of **what gets compressed**:
- **direct** → final `SR` → `metrics.json`: `Bpp`, `PSNR_cmpref`
- **ADMM** → auxiliary `Z` → `metrics.json`: `Bpp_Z`, `PSNR_cmpref_Z`

Default: 10 BPP-diverse images (`*_direct_10` / `*_admm_10`).  
Set `USE_FULL = True` for full DIV2K dirs (Nina/Swin full + ADMM).

In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

PROJECT = Path("/gpfs/gpfs0/timofey.glukhikh/Science_Phan")
if not PROJECT.exists():
    PROJECT = Path("/gpfs/data/gpfs0/timofey.glukhikh/Science_Phan")

RUNS = PROJECT / "runs"
OUT_DIR = RUNS / "rd_compare"
OUT_DIR.mkdir(parents=True, exist_ok=True)

USE_FULL = False  # True → full DIV2K dirs
BACKBONE = "swin"  # "nina" | "swin"

if USE_FULL:
    ROOT_DIRECT = RUNS / ("bitrate_sr_nina_full" if BACKBONE == "nina" else "bitrate_sr_swin")
    ROOT_ADMM = RUNS / f"bitrate_sr_{BACKBONE}_admm"
else:
    ROOT_DIRECT = RUNS / f"bitrate_sr_{BACKBONE}_direct_10"
    ROOT_ADMM = RUNS / f"bitrate_sr_{BACKBONE}_admm_10"

LAMS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]

print("direct:", ROOT_DIRECT, "exists=", ROOT_DIRECT.exists())
print("admm:  ", ROOT_ADMM, "exists=", ROOT_ADMM.exists())

In [ ]:
def run_dir(root: Path, img: str, lam: float, method: str) -> Path:
    return root / f"{img}_{BACKBONE}_psnr35_lam{lam:g}_r4_{method}"


def list_imgs(root: Path, method: str) -> list[str]:
    if not root.exists():
        return []
    return sorted({
        p.name.split(f"_{BACKBONE}_")[0]
        for p in root.glob(f"img*_{BACKBONE}_psnr35_lam*_r4_{method}")
        if (p / "metrics.json").exists()
    })


def load_pair_rows():
    """One row per (img, λ) where both direct SR and ADMM Z metrics exist."""
    imgs = sorted(set(list_imgs(ROOT_DIRECT, "direct")) & set(list_imgs(ROOT_ADMM, "admm")))
    rows = []
    for img in imgs:
        for lam in LAMS:
            md = run_dir(ROOT_DIRECT, img, lam, "direct") / "metrics.json"
            ma = run_dir(ROOT_ADMM, img, lam, "admm") / "metrics.json"
            if not md.exists() or not ma.exists():
                continue
            d = json.loads(md.read_text())
            a = json.loads(ma.read_text())
            if a.get("Bpp_Z") is None or a.get("PSNR_cmpref_Z") is None:
                continue
            if d.get("Bpp") is None or d.get("PSNR_cmpref") is None:
                continue
            rows.append({
                "img": img,
                "lam": float(lam),
                "bpp_sr_direct": float(d["Bpp"]),
                "psnr_sr_direct": float(d["PSNR_cmpref"]),
                "bpp_z_admm": float(a["Bpp_Z"]),
                "psnr_z_admm": float(a["PSNR_cmpref_Z"]),
                # also SR of ADMM for reference
                "bpp_sr_admm": float(a["Bpp"]) if a.get("Bpp") is not None else None,
                "psnr_sr_admm": float(a["PSNR_cmpref"]) if a.get("PSNR_cmpref") is not None else None,
            })
    return imgs, rows


imgs, rows = load_pair_rows()
print(f"images with both methods: {len(imgs)}")
print(f"paired (img, λ) points: {len(rows)}")
if rows:
    print("example:", rows[0])

## Mean RD: compress(SR_direct) vs compress(Z_ADMM)

In [ ]:
def mean_by_lam(rows, bpp_key, psnr_key):
    by = defaultdict(list)
    for r in rows:
        by[r["lam"]].append((r[bpp_key], r[psnr_key]))
    lams = sorted(by)
    xs = [float(np.mean([p[0] for p in by[l]])) for l in lams]
    ys = [float(np.mean([p[1] for p in by[l]])) for l in lams]
    ns = [len(by[l]) for l in lams]
    return lams, xs, ys, ns


if not rows:
    print("No paired metrics — check BACKBONE / USE_FULL / that ADMM has Bpp_Z")
else:
    l_d, x_d, y_d, n_d = mean_by_lam(rows, "bpp_sr_direct", "psnr_sr_direct")
    l_z, x_z, y_z, n_z = mean_by_lam(rows, "bpp_z_admm", "psnr_z_admm")

    print(f"{'lam':>6} {'Bpp_SR_dir':>10} {'PSNR_dir':>9} {'Bpp_Z':>10} {'PSNR_Z':>9} {'n':>4}")
    for i, lam in enumerate(l_d):
        # match λ index in Z list
        j = l_z.index(lam) if lam in l_z else None
        if j is None:
            continue
        print(f"{lam:6.3g} {x_d[i]:10.4f} {y_d[i]:9.3f} {x_z[j]:10.4f} {y_z[j]:9.3f} {n_d[i]:4d}")

    fig, ax = plt.subplots(figsize=(7.5, 5))
    od = np.argsort(x_d)
    oz = np.argsort(x_z)
    ax.plot([x_d[i] for i in od], [y_d[i] for i in od], "o-", color="C0", markersize=8,
            label=f"direct: compress(SR)  n≈{max(n_d)}")
    ax.plot([x_z[i] for i in oz], [y_z[i] for i in oz], "s--", color="C1", markersize=8,
            label=f"ADMM: compress(Z)  n≈{max(n_z)}")
    for i in od:
        ax.annotate(f"λ={l_d[i]:g}", (x_d[i], y_d[i]), textcoords="offset points",
                    xytext=(4, 4), fontsize=7, color="C0")
    for i in oz:
        ax.annotate(f"λ={l_z[i]:g}", (x_z[i], y_z[i]), textcoords="offset points",
                    xytext=(4, -10), fontsize=7, color="C1")

    ax.set_xlabel("Bpp (mean over images)")
    ax.set_ylabel("PSNR vs GT compressed (mean)")
    scope = "full DIV2K" if USE_FULL else "10 images"
    ax.set_title(f"{BACKBONE.upper()} — SR(direct) vs Z(ADMM) after compression ({scope})")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = OUT_DIR / f"rd_{BACKBONE}_SR_direct_vs_Z_admm.png"
    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out}")
    plt.show()
    plt.close(fig)

## Optional: also plot ADMM compress(SR) (same run as Z)

In [ ]:
if rows and any(r["bpp_sr_admm"] is not None for r in rows):
    l_a, x_a, y_a, n_a = mean_by_lam(
        [r for r in rows if r["bpp_sr_admm"] is not None],
        "bpp_sr_admm",
        "psnr_sr_admm",
    )
    fig, ax = plt.subplots(figsize=(7.5, 5))
    for xs, ys, lams, fmt, color, lab in [
        (x_d, y_d, l_d, "o-", "C0", "direct compress(SR)"),
        (x_z, y_z, l_z, "s--", "C1", "ADMM compress(Z)"),
        (x_a, y_a, l_a, "^:.", "C2", "ADMM compress(SR)"),
    ]:
        order = np.argsort(xs)
        ax.plot([xs[i] for i in order], [ys[i] for i in order], fmt, color=color, markersize=8, label=lab)
    ax.set_xlabel("Bpp (mean over images)")
    ax.set_ylabel("PSNR vs GT compressed (mean)")
    ax.set_title(f"{BACKBONE.upper()} — SR(direct) / Z(ADMM) / SR(ADMM)")
    ax.grid(True, alpha=0.3)
    ax.legend()
    out = OUT_DIR / f"rd_{BACKBONE}_SR_direct_vs_Z_and_SR_admm.png"
    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out}")
    plt.show()
    plt.close(fig)
else:
    print("skip")

## Visual: GT | SR(direct) | Z(ADMM) for one λ

In [ ]:
LAM_SHOW = 0.5  # change λ here

show_imgs = imgs[: min(6, len(imgs))]
if not show_imgs:
    print("no images")
else:
    n = len(show_imgs)
    fig, axes = plt.subplots(n, 3, figsize=(9, 2.2 * n))
    if n == 1:
        axes = np.array([axes])
    for i, img in enumerate(show_imgs):
        paths = [
            ("GT", run_dir(ROOT_DIRECT, img, LAM_SHOW, "direct") / "GT.png"),
            ("SR direct", run_dir(ROOT_DIRECT, img, LAM_SHOW, "direct") / "SR.png"),
            ("Z ADMM", run_dir(ROOT_ADMM, img, LAM_SHOW, "admm") / "Z.png"),
        ]
        md = run_dir(ROOT_DIRECT, img, LAM_SHOW, "direct") / "metrics.json"
        ma = run_dir(ROOT_ADMM, img, LAM_SHOW, "admm") / "metrics.json"
        meta = ""
        if md.exists() and ma.exists():
            d, a = json.loads(md.read_text()), json.loads(ma.read_text())
            meta = (
                f"dir Bpp={d.get('Bpp', float('nan')):.3f} PSNR={d.get('PSNR_cmpref', float('nan')):.1f} | "
                f"Z Bpp={a.get('Bpp_Z', float('nan')):.3f} PSNR={a.get('PSNR_cmpref_Z', float('nan')):.1f}"
            )
        for j, (title, path) in enumerate(paths):
            ax = axes[i, j]
            if path.exists():
                ax.imshow(Image.open(path))
            else:
                ax.text(0.5, 0.5, "missing", ha="center", va="center")
            ax.set_xticks([])
            ax.set_yticks([])
            if i == 0:
                ax.set_title(title)
            if j == 0:
                ax.set_ylabel(f"{img}\n{meta}", fontsize=7)
    fig.suptitle(f"{BACKBONE} λ={LAM_SHOW:g}: GT | SR(direct) | Z(ADMM)", y=1.01)
    fig.tight_layout()
    plt.show()
    plt.close(fig)